# RAG Demo — Pipeline Walkthrough

A minimal, modular Retrieval-Augmented Generation pipeline. Each stage is a standalone module that reads from one folder under `data/` and writes to the next, so you can run, swap, or upgrade each block independently.

**Stages** (built incrementally):
1. **Document processing** — load raw files, extract to markdown ← *covered below*
2. **Chunker** — split markdown into overlapping chunks ← *covered below*
3. Embedder — turn chunks into vectors
4. Vector store — store and search vectors
5. Retriever — top-k chunks for a query
6. Answerer — Claude generates a cited answer

## Stage 1 — Document Processing

- **Input:** `data/raw/` — drop your PDFs, `.md`, or `.txt` files here.
- **Output:** `data/markdown/` — one `.md` per input file, with YAML frontmatter (`title`, `authors`, `source`).
- **Extractor:** `pymupdf4llm` for PDF text + `pymupdf` metadata; pass-through for native text/markdown.

Two entry points:
- `extract_doc(path)` — pure function, takes one file, returns a `Document`. Use it to inspect the result for a single file.
- `extract_docs(input, output)` — runs `extract_doc` over a folder and writes `.md` (with frontmatter) to disk.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from document_processing import extract_doc, extract_docs
from chunker import chunk_doc, chunk_docs

In [ ]:
RAW = ROOT / "data" / "raw"
MARKDOWN = ROOT / "data" / "markdown"
CHUNKS = ROOT / "data" / "chunks"
RAW.mkdir(parents=True, exist_ok=True)
MARKDOWN.mkdir(parents=True, exist_ok=True)
CHUNKS.mkdir(parents=True, exist_ok=True)

print("raw dir:     ", RAW)
print("markdown dir:", MARKDOWN)
print("chunks dir:  ", CHUNKS)

### What's in `data/raw/`?

If empty, drop a PDF (or `.md`/`.txt`) in there and re-run.

In [ ]:
sorted(p.name for p in RAW.iterdir() if p.is_file())

### Single-file extraction (pure function)

Run `extract_doc()` on one file and inspect the resulting `Document`.

In [ ]:
files = [p for p in sorted(RAW.iterdir()) if p.is_file()]
assert files, f"No files in {RAW}. Drop a PDF/.md/.txt and re-run."

sample = files[0]
doc = extract_doc(sample)

print("source:  ", doc.source.name)
print("metadata:", doc.metadata)
print("length:  ", len(doc.text), "chars")
print()
print("--- first 1500 chars ---")
print(doc.text[:1500])

### Batch run (stage runner)

`extract_docs` extracts every supported file in `data/raw/` and writes a corresponding `.md` (with YAML frontmatter) to `data/markdown/`.

In [ ]:
docs = extract_docs(RAW, MARKDOWN)
for d in docs:
    print(f"{d.source.name:40s} -> {len(d.text):>8} chars  {d.metadata}")

### Inspect the output folder

In [ ]:
for p in sorted(MARKDOWN.iterdir()):
    if p.is_file() and p.suffix == ".md":
        print(f"{p.name:40s} {p.stat().st_size:>8} bytes")

## Stage 2 — Chunker

- **Input:** `data/markdown/` — `.md` files with YAML frontmatter from stage 1.
- **Output:** `data/chunks/` — one `.jsonl` per source doc, one chunk per line.
- **Strategy:** fixed-size character windows with overlap (default `size=2000`, `overlap=200`).

Two entry points:
- `chunk_doc(doc)` — pure function over an in-memory `Document`.
- `chunk_docs(input, output)` — reads `.md` files via frontmatter, chunks, writes `.jsonl`.

In [ ]:
all_chunks = [c for d in docs for c in chunk_doc(d)]
print(f"{len(all_chunks)} chunks across {len(docs)} docs\n")

first = all_chunks[0]
print("first chunk:")
print(f"  id:       {first.id}")
print(f"  source:   {first.source}")
print(f"  index:    {first.index}")
print(f"  metadata: {first.metadata}")
print(f"  text[:300]:\n{first.text[:300]}")

### Batch run

`chunk_docs` reads every `.md` in `data/markdown/` (frontmatter + body), chunks each, and writes one `.jsonl` per doc to `data/chunks/`.

In [ ]:
batched = chunk_docs(MARKDOWN, CHUNKS)
print(f"wrote {len(batched)} chunks total\n")

for p in sorted(CHUNKS.iterdir()):
    if p.suffix == ".jsonl":
        print(f"{p.name:40s} {p.stat().st_size:>8} bytes")